<a href="https://colab.research.google.com/github/SY-256/llms-from-scratch/blob/main/notebooks/ch07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 指示に従うためのファインチューニング
- LLMのインストラクションファインチューニングプロセス
- インストラクションチューニングされたLLMを評価する

## 7.2 教師ありインストラクションチューニングのためのデータセット準備

In [ ]:
# データセットをダウンロードする
import json
import os
import urllib

def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)

    with open(file_path, "r") as file:
        data = json.load(file)

    return data

file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
print(f"Number of entries: {len(data)}")

In [ ]:
# サンプルエントリを確認
print(f"Example entry:\n {data[50]}")

In [ ]:
# 別のサンプルも確認
# inputは空の場合もある
print(f"Another example entry:\n {data[999]}")

In [ ]:
# プロンプトフォーマット関数を実装する
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    return instruction_text + input_text

In [ ]:
# サンプルテキストで確認
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"
print(model_input + desired_response)

In [ ]:
# inputフィールドが空のサンプルの場合
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"
print(model_input + desired_response)

In [ ]:
# データセットを分割する
train_portion = int(len(data) * 0.85) # 85%のデータを学習に使用
test_portion = int(len(data) * 0.1) # 10%のデータをテストに使用
val_portion = len(data) - train_portion - test_portion # 残り5%のデータを検証用に使用

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

print(f"Training set length: {len(train_data)}")
print(f"Test set length: {len(test_data)}")
print(f"Validation set length: {len(val_data)}")

## 7.3 データを訓練バッチにまとめる
- サンプルデータを同じ長さに効率良くパディングする方法
- サンプルのリストをバッチにまとめるためにデフォルトのcollate関数を使用する
- collate関数は個々のデータサンプルのリストを1つのバッチにまとめて、訓練中にモデルが効率よく処理できるようにする役割を果たす

In [ ]:
# 指示データセットクラスを実装
import torch
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            # テキストを事前トークン化
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [ ]:
# '<|endoftext|>'を使ってパディングするためにトークンIDを取得
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

In [ ]:
# バッチ全体ではなく各バッチ内で長さを揃える
# 無駄なパディングを最小限に抑えられる

def custom_collate_draft_1(batch, pad_token_id=50256, device="cpu"):
    batch_max_length = max(len(item)+1 for item in batch) # バッチ内で最も長いシーケンスを特定
    inputs_lst = []

    for item in batch: # 入力のパディングと準備
        new_item = item.copy()
        new_item += [pad_token_id]

        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1]) # paddingに追加した余分なパディングトークンを削除
        inputs_lst.append(inputs)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    return inputs_tensor

In [ ]:
# サンプルで確認
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = (
    inputs_1,
    inputs_2,
    inputs_3
)

print(custom_collate_draft_1(batch))

In [ ]:
# 入力トークンIDからターゲットトークンIDを作成
def custom_collate_draft_2(batch, pad_token_id=50256, device="cpu"):
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        padded = (
            new_item + [pad_token_id] * (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1]) # 入力の最後のトークンを切り捨て
        targets = torch.tensor(padded[1:]) # ターゲットのために右に1つシフト
        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

In [ ]:
# サンプルで確認
inputs, targets = custom_collate_draft_2(batch)
print(inputs)
print(targets)

パディングトークンを訓練から除外できるように処理を行う->意味のあるデータだけが学習に使用されるようにする（分類モデルの場合は最後の出力トークンに基づいてモデルを訓練すれば良かったので、この点について考慮する必要がなかった）

ターゲットシーケンス側にはeotトークンID(50256)を1つ残しておくことで、指示に対する応答を生成するタイミングをLLMが学習できるようになり、生成された応答が完全であるというも印としてeotトークンが使われるようになる

In [ ]:
# カスタムcollate関数を実装する
# allowd_max_length=必要に応じてサンプルの長さを制御する（gpt2のコンテキストサイズ1,024トークンを超える場合に利用など）
def custom_collate_fn(batch, pad_token_id=50256, ignore_index=-100, allowed_max_length=None, device="cpu"):

    batch_max_length = max(len(item) + 1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]

        padded = (
            new_item + [pad_token_id] * (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])

        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

In [ ]:
# サンプルで確認
inputs, targets = custom_collate_fn(batch)
print(inputs)
print(targets)

CrossEntoryはデフォルトでindex=-100のものを無視するようになっている

そのため、-100のインデックスを設定して除外することで損失計算（モデル学習）に使用されなくなる

## 7.4 指示データセット用のデータローダーを作成する

In [ ]:
# GPUデバイス設定
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# 新しい関数を作成
from functools import partial

customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    allowed_max_length=1024
)

In [ ]:
# データローダーを初期化する
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8
torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

In [ ]:
# バッチサイズを調べる
print("Train loader: ")
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)

各バッチでトークンサイズが異なっている -> collate関数のおかげ

## 7.5 事前学習済みのLLMを読み込む

In [ ]:
# 事前学習済みモデル(355M)を読み込む

In [ ]:
! git clone https://github.com/rasbt/LLMs-from-scratch.git

In [ ]:

import sys
sys.path.append('/content/LLMs-from-scratch/ch07/01_main-chapter-code')

In [ ]:
from gpt_download import download_and_load_gpt2
from previous_chapters import GPTModel, load_weights_into_gpt

BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,
    "qkv_bias": True
}

model_config = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-medium (355M)"
BASE_CONFIG.update(model_config[CHOOSE_MODEL])

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")

settings, params = download_and_load_gpt2(
    model_size=model_size,
    models_dir="gpt2"
)

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval();

In [ ]:
# ファインチューニングを行う前の性能
torch.manual_seed(123)

input_text = format_input(val_data[0])
print(input_text)

In [ ]:
# モデルの応答を生成
from previous_chapters import generate, text_to_token_ids, token_ids_to_text

token_ids = generate(
    model=model,
    idx=text_to_token_ids(input_text, tokenizer),
    max_new_tokens=35,
    context_size=BASE_CONFIG["context_length"],
    eos_id=50256 # パディングトークンのトークンID
)
generated_text = token_ids_to_text(token_ids, tokenizer)

In [ ]:
# generate関数は入力と出力を結合したテキストを返す
# モデルの応答したテキストを切り離す(入力テキストと同じ長さ分だけ先頭から取り除く)
response_text = generated_text[len(input_text):].strip()
print(response_text)

## 7.6 指示データでのLLMのファインチューニング

In [ ]:
# 誤差関数と訓練関数
from previous_chapters import calc_loss_loader, train_model_simple

model.to(device)
torch.manual_seed(123)

with torch.no_grad():
    train_loss = calc_loss_loader(
        train_loader, model, device, num_batches=5
    )
    val_loss = calc_loss_loader(
        val_loader, model, device, num_batches=5
    )

print(f"Training loss: {train_loss}")
print(f"Validation loss: {val_loss}")

In [ ]:
# 事前学習済みLLMのインストラクションチューニング
import time

start_time = time.time()
torch.manual_seed(123)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=5e-5, weight_decay=0.1
)
num_epochs = 2

train_losses, val_losses, token_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context=format_input(val_data[0]), tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

In [ ]:
# 訓練と検証の誤差曲線を描画
from previous_chapters import plot_losses

epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, token_seen, train_losses, val_losses)

## 7.7 応答の抽出と保存

In [ ]:
# 応答抽出
torch.manual_seed(123)

for entry in test_data[:3]:
    input_text = format_input(entry)
    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)

    response_text = (
        generated_text[len(input_text):].replace("### Response:", "").strip()
    )

    print(input_text)
    print(f"\nCorrect response:\n>> {entry['output']}")
    print(f"\nModel response\n>> {response_text.strip()}")
    print("----------------------------------------------")

In [ ]:
from IPython.core.magics.basic import indent
# LLMの性能評価
# テストデータセットの応答を生成する
from tqdm import tqdm

for i, entry in tqdm(enumerate(test_data), total=len(test_data)):
    input_text = format_input(entry)

    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)

    response_text = (
        generated_text[len(input_text):].replace("### Response:", "").strip()
    )
    test_data[i]["model_response"] = response_text

with open("instruction-data-with-response.json", "w") as file:
    json.dump(test_data, file, indent=4)

In [ ]:
# サンプルのエントリ確認
print(test_data[0])

In [ ]:
# モデルの保存
import re

file_name = f"{re.sub(r'[ ()]', '', CHOOSE_MODEL) }-sft.pth"
torch.save(model.state_dict(), file_name)
print(f"Model saved as {file_name}")

## 7.8 ファインチューニングしたLLMを評価する
- 別の大規模なLLMを使って評価する

In [ ]:
!apt-get install -y zstd

!apt-get install -y lshw lspci

!nvidia-smi

!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
import time
import os

os.environ["PATH"] += ":/usr/local/bin"

proc = subprocess.Popen(
    ["/usr/local/bin/ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)
print("Ollama server started, PID:", proc.pid)

In [ ]:
import subprocess

result = subprocess.run(
    ["/usr/local/bin/ollama", "pull", "llama3.2:1b"],
    capture_output=True, text=True
)
print(result.stdout)

result = subprocess.run(
    ["/usr/local/bin/ollama", "run", "llama3.2:1b", "Say hello"],
    capture_output=True, text=True
)
print(result.stdout)

In [ ]:
!ollama pull llama3.1:8b

In [ ]:
import psutil

def check_if_running(process_name):
    running = False
    for proc in psutil.process_iter(["name"]):
        if process_name in proc.info["name"]:
            running = True
            break
    return running

ollama_running = check_if_running("ollama")

if not ollama_running:
    raise RuntimeError("Ollama nor running. Launch ollama before processing.")
print(f"Ollama running: {check_if_running("ollama")}")

In [ ]:
import urllib.request
import json

def query_model(prompt, model="llama3.1:8b", url="http://localhost:11434/api/chat"):

    data = {
        "model": model,
        "messages": [  # ← 修正
            {"role": "user", "content": prompt}
        ],
        "options": {
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048
        }
    }

    payload = json.dumps(data).encode("utf-8")

    request = urllib.request.Request(
        url,
        data=payload,
        method="POST"
    )

    request.add_header("Content-Type", "application/json")

    response_data = ""
    with urllib.request.urlopen(request) as response:
        while True:
            line = response.readline().decode("utf-8")
            if not line:
                break
            response_json = json.loads(line)
            response_data += response_json["message"]["content"]

    return response_data

In [ ]:
model = "llama3.1:8b"
result = query_model("What do Llamas eat?", model)
print(result)

In [ ]:
# テストデータの3件をサンプルに評価
for entry in test_data[:3]:
    prompt = (
        f"Given the input `{format_input(entry)}` "
        f"and correct output `{entry['output']}`, "
        f"score the model response `{entry['model_response']}`"
        f" on a scale from 0 to 100, where 100 is the best score. "
    )
    print("\nDataset response:")
    print(f">> {entry['output']}")
    print(f"\nModel response")
    print(f">> {entry['model_response']}")
    print(f"\nScore:")
    print(f">> {query_model(prompt)}")
    print("\n-----------------------------------------")

In [ ]:
# インストラクションチューニングされたLLMを評価する
def generate_model_scores(json_data, json_key, model="llama3.1:8b"):
    scores = []
    for entry in tqdm(json_data, desc="Scoring entries"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry['model_response']}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only."
        )
        score = query_model(prompt, model)
        try:
            scores.append(int(score))
        except ValueError:
            print(f"Could not convert score: {score}")
            continue

    return scores

In [ ]:
# テストデータ全体に対するスコアリング
scores = generate_model_scores(test_data, "model_response")
print(f"Number of scores: {len(scores)} of {len(test_data)}")
print(f"Average score: {sum(scores) / len(scores):.2f}\n")